# 01 — Validazione del Risk Engine vs target AlgoEagle

Confronta il `risk_panel` calcolato dal motore con i valori target estratti dalla
slide *"Analysis of Coherence"* (`data/sample/algoeagle_risk_targets.csv`).

⚠️ **Non si cerca un match esatto**: input, universo e finestra dei target non
sono noti. La validazione finanziaria verifica che le **relazioni di coerenza**
siano corrette (es. `CVaR ≥ VaR`, `EDaR ≥ CDaR ≥ DaR`, `MDD ≥ ADD`) e che gli
**ordini di grandezza** tornino. La validazione fine si fa in chat di progetto.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from aa_engine.data import PriceData
from aa_engine.risk import compute_measure, risk_panel

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE = ROOT / "data" / "sample"
targets = pd.read_csv(SAMPLE / "algoeagle_risk_targets.csv")
targets

## Portafoglio campione

In assenza dei prezzi originali AlgoEagle costruiamo un portafoglio multi-asset
sintetico (5 strumenti) con drawdown e code realistici, simile a un profilo
*Medium Risk*. Sostituibile con prezzi reali via `StaticFileProvider`.

In [ ]:
rng = np.random.default_rng(42)
dates = pd.bdate_range("2018-01-01", periods=1500)
tickers = ["EQ_US", "EQ_EU", "BOND_US", "GOLD", "OIL"]

# fattore di mercato comune + idiosincratico => correlazioni e code non banali
mkt = rng.standard_t(df=5, size=len(dates)) * 0.008
betas = np.array([1.1, 1.0, -0.2, 0.3, 0.9])
idio = rng.normal(0, 0.006, size=(len(dates), len(tickers)))
rets = mkt[:, None] * betas[None, :] + idio + 0.0002
prices = pd.DataFrame(100 * (1 + rets).cumprod(axis=0), index=dates, columns=tickers)

weights = pd.Series([0.35, 0.25, 0.30, 0.05, 0.05], index=tickers)
data = PriceData.from_prices(prices)
panel = risk_panel(data.returns, weights)
panel

## Confronto ordine di grandezza vs target

In [ ]:
name_to_code = {
    "Standard Deviation": "MV", "Mean Absolute Deviation": "MAD",
    "Semi Standard Deviation": "MSV", "Value at Risk": "VaR",
    "Conditional Value at Risk": "CVaR", "Entropic Value at Risk": "EVaR",
    "Tail Gini of Losses": "TG", "Worst Realization": "WR",
    "Skewness": "SKEW", "Kurtosis": "KT", "Ulcer Index": "UCI",
    "Average Drawdown": "ADD", "Drawdown at Risk": "DaR",
    "Conditional Drawdown at Risk": "CDaR", "Entropic Drawdown at Risk": "EDaR",
    "Relativistic Drawdown at Risk": "RLDaR", "Max Drawdown": "MDD",
}
by_code = panel.set_index("measure")["value"]
rows = []
for _, t in targets.iterrows():
    code = name_to_code.get(t["measure"])
    if code is None or code not in by_code.index:
        continue
    model_pct = by_code[code] * (1 if code in ("SKEW", "KT") else 100)
    rows.append({"measure": t["measure"], "code": code,
                 "target": t["value"], "model": round(model_pct, 2)})
pd.DataFrame(rows)

## Check di coerenza (devono essere tutti True)

In [ ]:
rets = data.returns
checks = {
    "CVaR >= VaR":  compute_measure(rets, weights, "CVaR") >= compute_measure(rets, weights, "VaR"),
    "EVaR >= CVaR": compute_measure(rets, weights, "EVaR") >= compute_measure(rets, weights, "CVaR"),
    "WR  >= EVaR":  compute_measure(rets, weights, "WR")   >= compute_measure(rets, weights, "EVaR"),
    "CDaR >= DaR":  compute_measure(rets, weights, "CDaR") >= compute_measure(rets, weights, "DaR"),
    "EDaR >= CDaR": compute_measure(rets, weights, "EDaR") >= compute_measure(rets, weights, "CDaR"),
    "MDD  >= ADD":  compute_measure(rets, weights, "MDD")  >= compute_measure(rets, weights, "ADD"),
    "MV   >= MSV":  compute_measure(rets, weights, "MV")   >= compute_measure(rets, weights, "MSV"),
}
assert all(checks.values()), checks
checks